# Demonstração técnica do pipeline SIH/SUS (AV1)

Este notebook prova, passo a passo, que as três etapas obrigatórias da AV1 estão implementadas e funcionando. Cada bloco abaixo mostra, com código rodando, uma das etapas do pipeline de dados.

**Pergunta de pesquisa:**
> Como as ondas do COVID-19 impactaram o volume de internações e a mortalidade hospitalar no SUS-SP entre 2020 e 2023?

**Quatro blocos, alinhados com as etapas da AV1:**

1. **Ingestão (bronze).** Leitura de um `.dbc` individual pelo `src/ingest_bronze.py`.
2. **Transformação (silver).** Seleção de campos, tipagem e enriquecimento pelo `src/transform_silver.py`.
3. **Qualidade dos dados.** Nulos e estatísticas básicas dos campos relevantes.
4. **Silver consolidada.** Carrega o Parquet final e confere totais de internações, COVID e óbitos.

> As visualizações exploratórias que respondem à pergunta de pesquisa ficam no notebook `02_silver_visualizacoes.ipynb`.

**Fonte:** SIH/SUS, AIH Reduzida, publicada pelo DATASUS.  
**Recorte:** Estado de São Paulo, janeiro de 2020 a dezembro de 2023.  
**Arquivos na bronze:** 48 `.dbc`.  
**Silver gerada:** 9.635.546 registros.

## 0. Configuração do ambiente

In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path(os.getcwd())
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from src.ingest_bronze import read_dbc_raw, list_dbc_files
from src.transform_silver import (
    select_fields, clean_dtypes,
    add_covid_flag, add_time_columns, add_age_group,
    COVID_CIDS,
)

BRONZE_DIR  = ROOT / 'data' / 'bronze' / 'sihsus' / 'SP'
SILVER_FILE = ROOT / 'data' / 'silver' / 'sihsus_sp.parquet'

print(f'Raiz do projeto: {ROOT}')
print(f'Silver disponível: {SILVER_FILE.exists()}')

## 1. Ingestão: leitura de um arquivo `.dbc` (bronze)

In [ ]:
# Lista os .dbc disponíveis na camada bronze
files = list_dbc_files(BRONZE_DIR)
print(f'Total de arquivos .dbc: {len(files)}')
print(f'Primeiro: {files[0].name}  |  Último: {files[-1].name}')
print()

# Lê um arquivo de exemplo: abril de 2021, mês de pico da segunda onda em SP
print('Lendo RDSP2104.dbc (abril de 2021)...')
df_bruto = read_dbc_raw(BRONZE_DIR / 'RDSP2104.dbc')
print(f'Registros lidos: {len(df_bruto):,}')
print(f'Colunas brutas:  {len(df_bruto.columns)}  (todos os campos do .dbc original)')

## 2. Transformação (silver): seleção, tipagem e enriquecimento

As células abaixo aplicam as funções do `src/transform_silver.py` sobre o arquivo de exemplo. Primeiro, seleção dos 17 campos relevantes e tipagem das colunas. Depois, o enriquecimento linha a linha: flag de COVID, colunas de tempo e faixa etária.

In [ ]:
# Seleção dos 17 campos relevantes e tipagem das colunas
df_sample = select_fields(df_bruto)
df_sample = clean_dtypes(df_sample)

print(f'Campos selecionados (silver): {len(df_sample.columns)}')
print()
print('Tipos finais após clean_dtypes:')
print(df_sample.dtypes.to_string())

### Nota sobre o CID do COVID no SIH/SUS

O SIH/SUS registra COVID-19 como **`B342`** (infecção por coronavírus NE), e não pelos códigos `U07.1` e `U07.2` do CID-10 oficial. Por isso, o `add_covid_flag` usa `B342` como filtro. Esta é uma particularidade da base que precisou ser descoberta durante a exploração e que justifica a escolha feita no pipeline da silver.

In [ ]:
# Enriquecimento: flag COVID, colunas de tempo e faixa etária
df_sample = add_covid_flag(df_sample)
df_sample = add_time_columns(df_sample)
df_sample = add_age_group(df_sample)

print(f'CID usado para COVID: {COVID_CIDS}')
n_covid = int(df_sample['is_covid'].sum())
n_total = len(df_sample)
print(f'Abril/2021: {n_covid:,} internações COVID de {n_total:,} totais ({100*n_covid/n_total:.1f}%)')
print()
print('Amostra das colunas derivadas:')
print(df_sample[['DT_INTER', 'ano', 'mes', 'ano_mes', 'faixa_etaria', 'is_covid']].head(5).to_string())

## 3. Qualidade dos dados

A silver é garantida sem nulos e sem duplicatas pelo `audit_quality`. As células abaixo provam isso sobre o arquivo de exemplo, antes da consolidação final.

In [ ]:
# Nulos por coluna no arquivo de exemplo
nulls = df_sample.isnull().sum()
resultado = nulls[nulls > 0]
print('Campos com nulos:')
if resultado.empty:
    print('  nenhum campo nulo neste arquivo')
else:
    print(resultado.to_string())

print()
print('Estatísticas básicas dos principais campos numéricos:')
df_sample[['DIAS_PERM', 'UTI_MES_TO', 'IDADE', 'MORTE']].describe().round(2)

## 4. Silver consolidada: carregar o Parquet final

A silver completa reúne os 48 arquivos mensais já processados pelo `transform_silver.py`. A célula abaixo carrega o Parquet final e confirma os totais de internações, COVID e óbitos. Este é o insumo que alimenta o notebook de visualizações.

In [ ]:
df_silver = pd.read_parquet(SILVER_FILE)

print('=== Resumo da camada silver ===')
print(f'Total de registros:  {len(df_silver):,}')
print(f'Período:             {df_silver["DT_INTER"].min().date()} a {df_silver["DT_INTER"].max().date()}')
print(f'Tamanho em memória:  {df_silver.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()

n_covid = int(df_silver['is_covid'].sum())
n_obito = int(df_silver['MORTE'].sum())
print('=== Validação contra a pergunta de pesquisa ===')
print(f'Internações COVID (CID B342): {n_covid:,} ({100*n_covid/len(df_silver):.1f}% das internações)')
print(f'Óbitos hospitalares totais:   {n_obito:,} ({100*n_obito/len(df_silver):.1f}% das internações)')
print()
print('Registros por ano de internação:')
print(df_silver['DT_INTER'].dt.year.value_counts().sort_index().to_string())